<a href="https://colab.research.google.com/github/amanda-carvalhosc/otimizacao-logistica-rj/blob/main/analise_e_limpeza_logistica.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

# Lendo o arquivo do Excel que você acabou de subir na pastinha
df = pd.read_excel('entregas_bruto.xlsx')

# Mostrando a tabela na tela
df


,ID_Entrega,Motorista,Regiao_Destino,Distancia_KM,Custo_Combustivel,Status_Entrega,Capacidade_Veiculo_KG,Peso_Carga_KG
0,ENT-001,Carlos Silva,Sao Goncalo,25,120.5,Entregue,1000,850
1,ENT-002,Mariana Souza,Niteroi,42,210.0,NaN,1500,1600
2,ENT-003,Carlos Silva,S茫o Gon莽alo,18,85.0,Entregue,1000,900
3,ENT-004,Roberto Lima,Rio de Janeiro,65,NaN,Em tr芒nsito,2000,1100
4,ENT-005,mariana souza,Niteroi,38,190.0,Cancelado,1500,400
5,ENT-006,Roberto Lima,Rio de Janeiro,70,320.0,Entregue,2000,2150
6,ENT-007,Carlos Silva,Sao Goncalo,12,60.0,Entregue,1000,300


In [ ]:
import pandas as pd
import numpy as np

# 1. Ler o arquivo Excel
df = pd.read_excel('entregas_bruto.xlsx')

# 2. Arrumar os nomes dos motoristas
df['Motorista'] = df['Motorista'].astype(str).str.strip().str.title()

# 3. SUPER TRATAMENTO DE REGIÃO: Procura por "Sao" ou "São" e força virar "São Gonçalo"
# Também limpa Niterói e Rio de Janeiro de espaços invisíveis
df['Regiao_Destino'] = df['Regiao_Destino'].astype(str).str.strip()
df.loc[df['Regiao_Destino'].str.contains('Sao|São|Gon', case=False, na=False), 'Regiao_Destino'] = 'São Gonçalo'
df.loc[df['Regiao_Destino'].str.contains('Nit', case=False, na=False), 'Regiao_Destino'] = 'Niterói'
df.loc[df['Regiao_Destino'].str.contains('Rio|Janeiro', case=False, na=False), 'Regiao_Destino'] = 'Rio de Janeiro'

# 4. Forçar os números a serem lidos corretamente pelo Python (limpando qualquer sujeira de texto)
df['Distancia_KM'] = pd.to_numeric(df['Distancia_KM'], errors='coerce')
df['Peso_Carga_KG'] = pd.to_numeric(df['Peso_Carga_KG'], errors='coerce')
df['Capacidade_Veiculo_KG'] = pd.to_numeric(df['Capacidade_Veiculo_KG'], errors='coerce')

# 5. Tratar o NaN do combustível baseado na distância real corrigida
custo_por_km_medio = (df['Custo_Combustivel'] / df['Distancia_KM']).mean()
df['Custo_Combustivel'] = df['Custo_Combustivel'].fillna(df['Distancia_KM'] * custo_por_km_medio)
df['Custo_Combustivel'] = df['Custo_Combustivel'].round(2)

# 6. Tratar o NaN do Status da Entrega
df['Status_Entrega'] = df['Status_Entrega'].fillna('Não Informado').str.strip()

print("--- PLANILHA TOTALMENTE CORRIGIDA E LIMPA ---")
df


--- PLANILHA TOTALMENTE CORRIGIDA E LIMPA ---


,ID_Entrega,Motorista,Regiao_Destino,Distancia_KM,Custo_Combustivel,Status_Entrega,Capacidade_Veiculo_KG,Peso_Carga_KG
0,ENT-001,Carlos Silva,São Gonçalo,25,120.5,Entregue,1000,850
1,ENT-002,Mariana Souza,Niterói,42,210.0,Não Informado,1500,1600
2,ENT-003,Carlos Silva,São Gonçalo,18,85.0,Entregue,1000,900
3,ENT-004,Roberto Lima,Rio de Janeiro,65,315.4,Em tr芒nsito,2000,1100
4,ENT-005,Mariana Souza,Niterói,38,190.0,Cancelado,1500,400
5,ENT-006,Roberto Lima,Rio de Janeiro,70,320.0,Entregue,2000,2150
6,ENT-007,Carlos Silva,São Gonçalo,12,60.0,Entregue,1000,300


In [ ]:
print("=== RELATÓRIO DE LOGÍSTICA PARA O CHEFE ===\n")

# Garante mais uma vez que qualquer variação de 'São Gonçalo' seja corrigida antes de calcular
df['Regiao_Destino'] = df['Regiao_Destino'].astype(str).str.strip()
df.loc[df['Regiao_Destino'].str.contains('Sao|São|Gon', case=False, na=False), 'Regiao_Destino'] = 'São Gonçalo'

# -------------------------------------------------------------------------
# Pergunta 1: Custo total atual de cada transporte (Soma total do combustível gasto)
# -------------------------------------------------------------------------
custo_total = df['Custo_Combustivel'].sum()
print(f"1. Custo Total Atual de Transporte da Filial: R$ {custo_total:,.2f}\n")

# -------------------------------------------------------------------------
# Pergunta 2: Quais motoristas estão sobrecarregados (Peso maior que a Capacidade)
# -------------------------------------------------------------------------
print("2. Verificação de Sobrecarga nos Veículos:")
# Criamos uma regra para achar quem levou mais peso do que a capacidade do carro permite
motoristas_sobrecarregados = df[df['Peso_Carga_KG'] > df['Capacidade_Veiculo_KG']]

if len(motoristas_sobrecarregados) > 0:
    for index, linha in motoristas_sobrecarregados.iterrows():
        print(f"   ⚠️ ALERTA: O motorista {linha['Motorista']} (ID: {linha['ID_Entrega']}) está SOBRECARREGADO!")
        print(f"      -> Carga real: {linha['Peso_Carga_KG']} KG | Capacidade máxima: {linha['Capacidade_Veiculo_KG']} KG")
else:
    print("   -> Sucesso: Nenhum motorista está trabalhando acima da capacidade hoje.")
print("\n")

# -------------------------------------------------------------------------
# Pergunta 3: Qual região está gerando mais custos de frete?
# -------------------------------------------------------------------------
print("3. Análise de Custo de Frete por Região:")
# Agrupamos as regiões e somamos o combustível de cada uma
custo_por_regiao = df.groupby('Regiao_Destino')['Custo_Combustivel'].sum().reset_index()

# Mostra o custo de cada região na tela
for index, linha in custo_por_regiao.iterrows():
    print(f"   -> {linha['Regiao_Destino']}: R$ {linha['Custo_Combustivel']:,.2f}")

# Identifica automaticamente qual é a região com o maior valor
regiao_mais_cara = custo_por_regiao.sort_values(by='Custo_Combustivel', ascending=False).iloc[0]
print(f"\n📢 CONCLUSÃO: A região que gera MAIS CUSTO para a empresa é: {regiao_mais_cara['Regiao_Destino']} (R$ {regiao_mais_cara['Custo_Combustivel']:,.2f})")


=== RELATÓRIO DE LOGÍSTICA PARA O CHEFE ===

1. Custo Total Atual de Transporte da Filial: R$ 1,300.90

2. Verificação de Sobrecarga nos Veículos:
   ⚠️ ALERTA: O motorista Mariana Souza (ID: ENT-002) está SOBRECARREGADO!
      -> Carga real: 1600 KG | Capacidade máxima: 1500 KG
   ⚠️ ALERTA: O motorista Roberto Lima (ID: ENT-006) está SOBRECARREGADO!
      -> Carga real: 2150 KG | Capacidade máxima: 2000 KG


3. Análise de Custo de Frete por Região:
   -> Niterói: R$ 400.00
   -> Rio de Janeiro: R$ 635.40
   -> São Gonçalo: R$ 265.50

📢 CONCLUSÃO: A região que gera MAIS CUSTO para a empresa é: Rio de Janeiro (R$ 635.40)


In [ ]:
# Salva a tabela com todos os nomes e números corrigidos em um novo arquivo de Excel
df.to_excel('entregas_limpas_operacao.xlsx', index=False)
print("🚀 Sucesso! O arquivo 'entregas_limpas_operacao.xlsx' foi gerado.")
print("Agora é só ir na pastinha do lado esquerdo do Colab e clicar nos 3 pontinhos do arquivo para fazer o Download!")


🚀 Sucesso! O arquivo 'entregas_limpas_operacao.xlsx' foi gerado.
Agora é só ir na pastinha do lado esquerdo do Colab e clicar nos 3 pontinhos do arquivo para fazer o Download!
